# MACE+Graph2Mat

This notebook will show you how to integrate a `MACE` model with `Graph2Mat` through the python API. Note that you can also use `MACE+Graph2Mat` through the Command Line Interface (CLI).

Prerequisites
-------------
Before reading this notebook, **make sure you have read the [notebook on computing a matrix](<./Computing a matrix.ipynb>) and [the notebook on batching](./Batching.ipynb)**, which introduce the basic concepts of `graph2mat` that we are going to assume are already known. Also **we will use exactly the same setup as in the batching notebook**, with the only difference that we will add target matrices to each structure.

In [1]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [ ]:
import numpy as np
import pandas as pd
import torch

# To load plotly templates for sisl visualization
import sisl.viz

from e3nn import o3

from graph2mat import (
    BasisConfiguration,
    PointBasis,
    BasisTableWithEdges,
    MatrixDataProcessor,
)
from graph2mat.bindings.torch import TorchBasisMatrixDataset, TorchBasisMatrixData

from graph2mat.bindings.e3nn import E3nnGraph2Mat

from graph2mat.tools.viz import plot_basis_matrix


from torch_geometric.loader import DataLoader

Generating a dataset
--------------------

We generate a dataset here just as we have done in the other notebooks.

In [ ]:
# The basis
point_1_row = PointBasis("A", R=2, basis="0e", basis_convention="spherical")  # "0e"
point_1_col = PointBasis("A", R=2, basis="2x0e", basis_convention="spherical")
point_2_row = PointBasis("B", R=5, basis="0e + 1o", basis_convention="spherical")
point_2_col = PointBasis("B", R=5, basis="2x0e + 1o", basis_convention="spherical")

this_basis = {'row': [point_1_row,  point_2_row], 'col': [point_1_col, point_2_col]}

# The basis table.
table = BasisTableWithEdges(this_basis)

# The data processor.
processor = MatrixDataProcessor(
    basis_table=table, symmetric_matrix=False,  # Matrix is not square
    sub_point_matrix=False
)

positions = np.array([[0, 0, 0], [6.0, 0, 0], [9.0, 0, 0]])

config1 = BasisConfiguration(
    point_types=["A", "B", "A"],
    positions=positions,
    basis=this_basis,
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

config2 = BasisConfiguration(
    point_types=["B", "A", "B"],
    positions=positions,
    basis=this_basis,
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

configs = [config1, config2]

dataset = TorchBasisMatrixDataset(configs, data_processor=processor)


loader = DataLoader(dataset, batch_size=2)

data = next(iter(loader))

Initializing a MACE model
-------------------------

We will now initialize a normal MACE model.

Note that you must have MACE installed, which you can do with:

```
pip install mace_torch
```

In [ ]:
from mace.modules import MACE, RealAgnosticResidualInteractionBlock

num_interactions = 3
hidden_irreps = o3.Irreps("1x0e + 1x1o")

mace_model = MACE(
    r_max=10,
    num_bessel=10,
    num_polynomial_cutoff=10,
    max_ell=2,  # 1,
    interaction_cls=RealAgnosticResidualInteractionBlock,
    interaction_cls_first=RealAgnosticResidualInteractionBlock,
    num_interactions=num_interactions,
    num_elements=2,
    hidden_irreps=hidden_irreps,
    MLP_irreps=o3.Irreps("2x0e"),
    atomic_energies=torch.tensor([0, 0]),
    avg_num_neighbors=2,
    atomic_numbers=[0, 1],
    correlation=2,
    gate=None,
)

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/mace/modules/blocks.py:312: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(atomic_energies, dtype=torch.get_default_dtype()),
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

Now, we can pass our data through the mace model. MACE outputs many things, but we are just interested in the node features, which we can get from the `"node_feats"` key.

In [ ]:
mace_output = mace_model(data)
mace_output["node_feats"]

tensor([[ 2.4548e-01,  0.0000e+00,  0.0000e+00,  1.9895e-04, -1.7979e-01,
          0.0000e+00,  0.0000e+00, -4.4004e-04,  3.8175e-02],
        [ 4.1981e-01,  0.0000e+00,  0.0000e+00, -8.7587e-05,  4.0425e-02,
          0.0000e+00,  0.0000e+00,  6.3208e-04, -2.8633e-02],
        [ 2.4413e-01,  0.0000e+00,  0.0000e+00, -1.0336e-03, -1.7886e-01,
          0.0000e+00,  0.0000e+00,  1.6616e-04,  3.7214e-02],
        [ 4.1830e-01,  0.0000e+00,  0.0000e+00, -1.6220e-05,  4.0248e-02,
          0.0000e+00,  0.0000e+00, -1.6040e-04, -2.0731e-02],
        [ 2.4420e-01,  0.0000e+00,  0.0000e+00,  8.3819e-04, -1.7869e-01,
          0.0000e+00,  0.0000e+00,  2.6878e-04,  3.6943e-02],
        [ 4.1949e-01,  0.0000e+00,  0.0000e+00,  1.0425e-04,  4.0349e-02,
          0.0000e+00,  0.0000e+00, -4.7078e-04, -2.6764e-02]],
       grad_fn=<CatBackward0>)

Our `Graph2Mat` model will take these node features and convert them to a matrix. Therefore we need to know what its irreps are, and then initialize the `Graph2Mat` module.

In [ ]:
# MACE outputs as node features the hidden irreps for each interaction, except
# in the last interaction, where it computes just scalar features.
mace_out_irreps = hidden_irreps * (num_interactions - 1) + str(hidden_irreps[0])

# Initialize the matrix model with this information
matrix_model = E3nnGraph2Mat(
    unique_basis=table,
    irreps=dict(node_feats_irreps=mace_out_irreps),
    symmetric=False,  # Matrix is not square
    # We would need to also implement passing the edge information in order to use
    # preprocessing_edges. As shown later, graph2mat can do this automatically for you.
    preprocessing_edges=None,
)

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/

Now, we can use the matrix model, passing the node features computed by MACE:

In [ ]:
node_labels, edge_labels = matrix_model(data=data, node_feats=mace_output["node_feats"])

In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0 1 0 1]
shapes:  [[1 4]
 [2 5]]
shapes_inv:  [[1 4]
 [2 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
len(indices):  66
indices:  [ 0  1  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25  2  3
 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45  4  5 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65]
In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [ 1 -1  1 -1  1 -1  1 -1  2 -2]
shapes:  [[1 1 4]
 [2 5 5]]
shapes_inv:  [[1 4 4]
 [2 2 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
len(indices):  92
indices:  [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 8

And plot the obtained matrices:

In [ ]:
matrices = processor.matrix_from_data(
    data,
    predictions={"node_labels": node_labels, "edge_labels": edge_labels},
)

for config, matrix in zip(configs, matrices):
    plot_basis_matrix(
        matrix,
        config,
        point_lines={"color": "black"},
        basis_lines={"color": "blue"},
        colorscale="temps",
        text=".2f",
        basis_labels=True,
    ).show()

In sparse.py _blockmatrix_coo_coords:
orbitals_row: [1, 4, 1]
orbitals_col: [2, 5, 2]
edge_index: [[0 1 2 1]
 [1 0 1 2]]
In sparse.py _blockmatrix_coo_coords:
orbitals_row: [4, 1, 4]
orbitals_col: [5, 2, 5]
edge_index: [[1 0 1 2 0 2]
 [0 1 2 1 2 0]]
[PointBasis(type='A', R=2, basis=((1, 0, 1),), basis_convention='spherical'), PointBasis(type='B', R=5, basis=((1, 0, 1), (1, 1, -1)), basis_convention='spherical')]


[PointBasis(type='A', R=2, basis=((1, 0, 1),), basis_convention='spherical'), PointBasis(type='B', R=5, basis=((1, 0, 1), (1, 1, -1)), basis_convention='spherical')]


Using MatrixMACE
----------------

If you don't want to handle the details of interacting `MACE` with `Graph2Mat`, you can also use `MatrixMACE`, which takes a mace model and wraps it to also output the `node_labels` and `edge_labels` corresponding to a matrix. 

Internally, it just initializes a `E3nnGraph2Mat` layer. However it can handle the interaction between `MACE` and `Graph2Mat` in more complex cases like having an extra preprocessing step for edges, which needs some extra inputs from MACE.

In [ ]:
from graph2mat.models import MatrixMACE
from graph2mat.bindings.e3nn import E3nnEdgeMessageBlock

In [ ]:
matrix_mace_model = MatrixMACE(
    mace_model,
    unique_basis=table,
    readout_per_interaction=True,
    edge_hidden_irreps=o3.Irreps("10x0e + 10x1o + 10x2e"),
    symmetric=False,
)

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/saru/anaconda3/envs/basisc/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning:

The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.

/home/saru/anaconda3/envs/basisc/lib/python3.1

The output of this model is MACE's output plus the `node_labels` and `edge_labels` for the predicted matrix:

In [ ]:
out = matrix_mace_model(data)


In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0 1 0 1]
shapes:  [[1 4]
 [2 5]]
shapes_inv:  [[1 4]
 [2 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
len(indices):  66
indices:  [ 0  1  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25  2  3
 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45  4  5 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65]
In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [ 1 -1  1 -1  1 -1  1 -1  2 -2]
shapes:  [[1 1 4]
 [2 5 5]]
shapes_inv:  [[1 4 4]
 [2 2 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
len(indices):  92
indices:  [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 8

{'energy': tensor([1.0543, 1.0004], grad_fn=<SumBackward1>),
 'node_energy': tensor([0.3761, 0.3056, 0.3726, 0.3194, 0.3721, 0.3089],
        grad_fn=<SumBackward1>),
 'contributions': tensor([[ 0.0000,  0.0000,  0.8202,  0.1449,  0.0893],
         [ 0.0000,  0.0000,  0.9758,  0.0447, -0.0201]],
        grad_fn=<StackBackward0>),
 'forces': None,
 'edge_forces': None,
 'virials': None,
 'stress': None,
 'atomic_virials': None,
 'atomic_stresses': None,
 'displacement': tensor([[[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],
 
         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]),
 'hessian': None,
 'node_feats': tensor([[ 2.4548e-01,  0.0000e+00,  0.0000e+00,  1.9895e-04, -1.7979e-01,
           0.0000e+00,  0.0000e+00, -4.4004e-04,  3.8175e-02],
         [ 4.1981e-01,  0.0000e+00,  0.0000e+00, -8.7587e-05,  4.0425e-02,
           0.0000e+00,  0.0000e+00,  6.3208e-04, -2.8633e-02],
         [ 2.4413e-01,  0.0000e+00,  0.0000e+00, -1.0336e-03, -1.7886

You can of course plot the predicted matrices:

In [ ]:
matrices = processor.matrix_from_data(data, predictions=out)

for config, matrix in zip(configs, matrices):
    plot_basis_matrix(
        matrix*1e5,
        config,
        point_lines={"color": "black"},
        basis_lines={"color": "blue"},
        colorscale="temps",
        text=".2f",
        basis_labels=True,
    ).show()

In sparse.py _blockmatrix_coo_coords:
orbitals_row: [1, 4, 1]
orbitals_col: [2, 5, 2]
edge_index: [[0 1 2 1]
 [1 0 1 2]]
In sparse.py _blockmatrix_coo_coords:
orbitals_row: [4, 1, 4]
orbitals_col: [5, 2, 5]
edge_index: [[1 0 1 2 0 2]
 [0 1 2 1 2 0]]
[PointBasis(type='A', R=2, basis=((1, 0, 1),), basis_convention='spherical'), PointBasis(type='B', R=5, basis=((1, 0, 1), (1, 1, -1)), basis_convention='spherical')]


[PointBasis(type='A', R=2, basis=((1, 0, 1),), basis_convention='spherical'), PointBasis(type='B', R=5, basis=((1, 0, 1), (1, 1, -1)), basis_convention='spherical')]


# Rotating matrix

A matrix should rotate equivariantly if we rotate the configuration given to mace via the Data. Let us try!

Thing sthat remain constant under rotation: the basis we defined and the thing sthat depend on it.

- table object: an processed object with the basis of each point basis (and the basis points point_1, point_2, point_3, point_4)
- data_processor object processor : has info of how to process the basis, i.e., information about the edges and pointers.
- mace_model and matrix_mace_model : it is just the architecture of the mace model, so we use the same model for both the original and rotated configurations. Matrixmace is just the matrixed version of the mace model, so it is also the same for both configurations.

In [ ]:
positions_rot = np.array([[0, 0, 0], [0, 6.0, 0], [0, 9.0, 0]])

config1_rot = BasisConfiguration(
    point_types=["A", "B", "A"],
    positions=positions_rot,  # changed positions to rotated ones
    basis=this_basis,
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

config2_rot = BasisConfiguration(
    point_types=["B", "A", "B"],
    positions=positions_rot,  # changed positions to rotated ones
    basis=this_basis,
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

configs_rot = [config1_rot, config2_rot]

dataset_rot = TorchBasisMatrixDataset(configs_rot, data_processor=processor)


loader_rot = DataLoader(dataset_rot, batch_size=2)

data_rot = next(iter(loader_rot))

In [ ]:
out_rot = matrix_mace_model(data_rot)

In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [0 1 0 1 0 1]
shapes:  [[1 4]
 [2 5]]
shapes_inv:  [[1 4]
 [2 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
len(indices):  66
indices:  [ 0  1  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25  2  3
 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45  4  5 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65]
In Graph2Mat _get_labels_resort_index: 
BEFORE CALLING get_labels_resorting_array
types:  [ 1 -1  1 -1  1 -1  1 -1  2 -2]
shapes:  [[1 1 4]
 [2 5 5]]
shapes_inv:  [[1 4 4]
 [2 2 5]]
transpose_neg:  False
kwargs:  {}
AFTER CALLING get_labels_resorting_array
len(indices):  92
indices:  [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 8

You can of course plot the predicted matrices:

In [ ]:
matrices_rot = processor.matrix_from_data(data_rot, predictions=out_rot)

for config, matrix in zip(configs_rot, matrices_rot):
    plot_basis_matrix(
        matrix*1e5,
        config,
        point_lines={"color": "black"},
        basis_lines={"color": "blue"},
        colorscale="temps",
        text=".2f",
        basis_labels=True,
    ).show()

In sparse.py _blockmatrix_coo_coords:
orbitals_row: [1, 4, 1]
orbitals_col: [2, 5, 2]
edge_index: [[0 1 2 1]
 [1 0 1 2]]
In sparse.py _blockmatrix_coo_coords:
orbitals_row: [4, 1, 4]
orbitals_col: [5, 2, 5]
edge_index: [[1 0 1 2 0 2]
 [0 1 2 1 2 0]]
[PointBasis(type='A', R=2, basis=((1, 0, 1),), basis_convention='spherical'), PointBasis(type='B', R=5, basis=((1, 0, 1), (1, 1, -1)), basis_convention='spherical')]


[PointBasis(type='A', R=2, basis=((1, 0, 1),), basis_convention='spherical'), PointBasis(type='B', R=5, basis=((1, 0, 1), (1, 1, -1)), basis_convention='spherical')]


In [ ]:
positions_rot2 = np.array([[0, 0, 0], [0, 0, 6.0], [0, 0, 9.0]])

config1_rot2 = BasisConfiguration(
    point_types=["A", "B", "A"],
    positions=positions_rot2,  # changed positions to rotated ones
    basis=[point_1, point_2, point_3, point_4],
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

config2_rot2 = BasisConfiguration(
    point_types=["B", "A", "B"],
    positions=positions_rot2,  # changed positions to rotated ones
    basis=[point_1, point_2, point_3, point_4],
    cell=np.eye(3) * 100,
    pbc=(False, False, False),
)

configs_rot2 = [config1_rot2, config2_rot2]

dataset_rot2 = TorchBasisMatrixDataset(configs_rot2, data_processor=processor)


loader_rot2 = DataLoader(dataset_rot2, batch_size=2)

data_rot2 = next(iter(loader_rot2))

NameError: name 'point_1' is not defined

In [ ]:
out_rot = matrix_mace_model(data_rot)

out_rot

{'energy': tensor([0.3526, 0.1785], grad_fn=<SumBackward1>),
 'node_energy': tensor([0.1747, 0.0031, 0.1748, 0.0004, 0.1748, 0.0033],
        grad_fn=<SumBackward1>),
 'contributions': tensor([[0.0000, 0.0000, 0.0645, 0.1906, 0.0975],
         [0.0000, 0.0000, 0.0328, 0.0974, 0.0484]], grad_fn=<StackBackward0>),
 'forces': None,
 'edge_forces': None,
 'virials': None,
 'stress': None,
 'atomic_virials': None,
 'atomic_stresses': None,
 'displacement': tensor([[[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],
 
         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]),
 'hessian': None,
 'node_feats': tensor([[ 1.3882e-01,  1.6603e-05,  0.0000e+00,  0.0000e+00, -1.3475e-01,
          -1.2804e-05,  0.0000e+00,  0.0000e+00, -1.6640e-01],
         [ 2.0248e-03, -1.3442e-02,  0.0000e+00,  0.0000e+00, -1.8957e-03,
           5.6720e-03,  0.0000e+00,  0.0000e+00, -4.3648e-03],
         [ 1.3882e-01, -3.2994e-05,  0.0000e+00,  0.0000e+00, -1.3478e-01,
           8

You can of course plot the predicted matrices:

In [ ]:
matrices_rot = processor.matrix_from_data(data_rot, predictions=out_rot)

for config, matrix in zip(configs_rot, matrices_rot):
    plot_basis_matrix(
        matrix,
        config,
        point_lines={"color": "black"},
        basis_lines={"color": "blue"},
        colorscale="temps",
        text=".2f",
        basis_labels=True,
    ).show()

Summary and next steps
----------------------

In this notebook we learned **how to interface MACE with Graph2Mat**.

The **next steps** could be:

- **Train a MACE+Graph2Mat model** following the steps in [this notebook](<./Fitting matrices.ipynb>), replacing the model by the `MACE+Graph2Mat` model.